In [1]:
# ============================================================
# Item-Item Co-Purchase Graph in TensorFlow 2
# End-to-end version with:
# - tf.data positive dataset
# - per-batch negative sampling
# - optional false-negative filtering
# - model.fit() style training
# - separate validation metrics at epoch end
# - manual weighted BCE for tf.keras compatibility
# ============================================================

import random
from typing import Dict, Set, Optional

import numpy as np
import pandas as pd
import tensorflow as tf
from scipy import sparse


# ------------------------------------------------------------
# 1) Reproducibility
# ------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


def build_copurchase_graph(
    df: pd.DataFrame,
    order_col: str = "order_id",
    item_col: str = "product_id",
    min_item_freq: int = 2,
    min_items_per_order: int = 5,
    max_items_per_order: int = 20,
    weight_mode: str = "lift",   # "cosine", "lift", "lift_shrink", "ppmi", "confidence"
    topk_neighbors: int = 200,
    shrink_alpha: float = 10.0,
    dtype=np.float32,
):
    work = df[[order_col, item_col]].dropna().copy()
    work[item_col] = work[item_col].astype(str)
    if work.empty:
        raise ValueError("No data after dropna")

    order_sizes = work.groupby(order_col, sort=False)[item_col].size()
    valid_orders = order_sizes[
        (order_sizes >= min_items_per_order) & (order_sizes <= max_items_per_order)
    ].index
    work = work[work[order_col].isin(valid_orders)]

    if work.empty:
        raise ValueError("No data left after basket-size filtering")

    item_freq_series = work.groupby(item_col, sort=False)[order_col].size()
    valid_items = item_freq_series[item_freq_series >= min_item_freq].index
    work = work[work[item_col].isin(valid_items)]

    if work.empty:
        raise ValueError("No data left after min_item_freq filtering")

    item_codes, item_uniques = pd.factorize(work[item_col], sort=False)
    order_codes, order_uniques = pd.factorize(work[order_col], sort=False)

    n_orders = len(order_uniques)
    n_items = len(item_uniques)

    data = np.ones(len(work), dtype=dtype)
    X = sparse.coo_matrix(
        (data, (order_codes.astype(np.int32), item_codes.astype(np.int32))),
        shape=(n_orders, n_items),
        dtype=dtype,
    ).tocsr()

    item_freq = np.asarray(X.sum(axis=0)).ravel().astype(np.float64)

    C = (X.T @ X).tocsr()
    C.setdiag(0)
    C.eliminate_zeros()

    rows = []
    total_orders = float(n_orders)

    indptr = C.indptr
    indices = C.indices
    counts = C.data.astype(np.float64, copy=False)

    supported_modes = {"cosine", "lift", "lift_shrink", "ppmi", "confidence"}
    if weight_mode not in supported_modes:
        raise ValueError(f"weight_mode must be one of: {sorted(supported_modes)}")

    for i in range(n_items):
        start, end = indptr[i], indptr[i + 1]
        if start == end:
            continue

        nbr_idx = indices[start:end]
        c_ij = counts[start:end]

        n_i = item_freq[i]
        n_j = item_freq[nbr_idx]

        # compute all metrics
        cosine = c_ij / np.sqrt(np.maximum(n_i * n_j, 1.0))
        lift = (c_ij * total_orders) / np.maximum(n_i * n_j, 1.0)

        shrink = c_ij / (c_ij + float(shrink_alpha))
        lift_shrink = lift * shrink

        p_ij = c_ij / total_orders
        p_i = n_i / total_orders
        p_j = n_j / total_orders
        pmi = np.log((p_ij + 1e-12) / (p_i * p_j + 1e-12))
        ppmi = np.maximum(pmi, 0.0)

        confidence = c_ij / np.maximum(n_i, 1.0)

        metric_map = {
            "cosine": cosine,
            "lift": lift,
            "lift_shrink": lift_shrink,
            "ppmi": ppmi,
            "confidence": confidence,
        }
        rank_w = metric_map[weight_mode]

        if len(rank_w) > topk_neighbors:
            top_local = np.argpartition(-rank_w, topk_neighbors - 1)[:topk_neighbors]

            nbr_idx = nbr_idx[top_local]
            c_ij = c_ij[top_local]

            cosine = cosine[top_local]
            lift = lift[top_local]
            shrink = shrink[top_local]
            lift_shrink = lift_shrink[top_local]
            ppmi = ppmi[top_local]
            confidence = confidence[top_local]
            rank_w = rank_w[top_local]

        order = np.lexsort((-c_ij, -rank_w))[::-1]

        src_item = item_uniques[i]
        for k in order:
            rows.append(
                (
                    src_item,
                    item_uniques[nbr_idx[k]],
                    float(rank_w[k]),          # selected ranking weight
                    float(cosine[k]),
                    float(lift[k]),
                    float(shrink[k]),
                    float(lift_shrink[k]),
                    float(ppmi[k]),
                    float(confidence[k]),
                    int(c_ij[k]),
                )
            )

    if not rows:
        raise ValueError("No graph edges created after sparse multiplication")

    edges_df = pd.DataFrame(
        rows,
        columns=[
            "src_item",
            "dst_item",
            "weight",          # active ranking weight based on weight_mode
            "cosine",
            "lift",
            "shrink",
            "lift_shrink",
            "ppmi",
            "confidence",
            "co-occurance",
        ],
    )

    item_freq_dict = pd.Series(item_freq, index=item_uniques).astype(int).to_dict()
    return edges_df, item_freq_dict
    
# ------------------------------------------------------------
# 3) Train/validation split
# ------------------------------------------------------------
def train_val_split_edges(
    edges_df: pd.DataFrame,
    val_frac: float = 0.2,
    seed: int = SEED,
):
    """
    Random edge split.
    For stronger evaluation, later you may want a basket-level or time-based split.
    """
    if not 0.0 < val_frac < 1.0:
        raise ValueError("val_frac must be between 0 and 1")

    n = len(edges_df)
    if n < 2:
        raise ValueError("Need at least 2 edges for train/validation split")

    rng = np.random.default_rng(seed)
    idx = np.arange(n)
    rng.shuffle(idx)

    n_val = max(1, int(n * val_frac))
    val_idx = idx[:n_val]
    train_idx = idx[n_val:]

    if len(train_idx) == 0:
        raise ValueError("Train split is empty after split")

    train_edges = edges_df.iloc[train_idx].reset_index(drop=True)
    val_edges = edges_df.iloc[val_idx].reset_index(drop=True)
    return train_edges, val_edges


# ------------------------------------------------------------
# 4) Vocabulary + arrays
# ------------------------------------------------------------
def build_item_vocab(edges_df: pd.DataFrame):
    items = pd.Index(
        pd.concat([edges_df["src_item"], edges_df["dst_item"]]).astype(str).unique()
    )
    item2idx = {item: idx for idx, item in enumerate(items)}
    idx2item = {idx: item for item, idx in item2idx.items()}
    return item2idx, idx2item


def edges_to_training_arrays(edges_df: pd.DataFrame, item2idx: dict):
    src = edges_df["src_item"].map(item2idx).to_numpy(dtype=np.int32)
    dst = edges_df["dst_item"].map(item2idx).to_numpy(dtype=np.int32)
    wgt = edges_df["weight"].to_numpy(dtype=np.float32)

    if len(wgt) > 0:
        wgt = wgt / (wgt.mean() + 1e-8)

    return src, dst, wgt


# ------------------------------------------------------------
# 5) Negative sampling distribution
# ------------------------------------------------------------
def build_negative_sampling_distribution(
    edges_df: pd.DataFrame,
    item2idx: dict,
    mode: str = "uniform",   # "popular" or "uniform"
    power: float = 0.75,
):
    if mode == "uniform":
        probs = np.ones(len(item2idx), dtype=np.float64)
        probs = probs / probs.sum()
        return probs.astype(np.float32)

    if mode == "popular":
        freq = edges_df["dst_item"].value_counts()
        probs = np.ones(len(item2idx), dtype=np.float64)

        for item, count in freq.items():
            probs[item2idx[item]] = float(count) ** power

        probs = probs / probs.sum()
        return probs.astype(np.float32)

    raise ValueError("mode must be one of: 'popular', 'uniform'")

# ------------------------------------------------------------
# 6) Optional false-negative filtering structures
# ------------------------------------------------------------
def build_positive_neighbor_sets(edges_df: pd.DataFrame, item2idx: dict) -> Dict[int, Set[int]]:
    neighbor_sets = {}
    for src_item, grp in edges_df.groupby("src_item"):
        src_idx = item2idx[src_item]
        dst_set = set(grp["dst_item"].map(item2idx).tolist())
        neighbor_sets[src_idx] = dst_set
    return neighbor_sets


# ------------------------------------------------------------
# 7) tf.data positive-only dataset
# ------------------------------------------------------------
def make_positive_tf_dataset(
    edges_df: pd.DataFrame,
    item2idx: dict,
    batch_size: int = 4096,
    shuffle: bool = True,
):
    pos_src, pos_dst, pos_wgt = edges_to_training_arrays(edges_df, item2idx)

    ds = tf.data.Dataset.from_tensor_slices({
        "src_item": pos_src.astype(np.int32),
        "pos_dst_item": pos_dst.astype(np.int32),
        "pos_weight": pos_wgt.astype(np.float32),
    })

    if shuffle:
        ds = ds.shuffle(
            buffer_size=min(len(pos_src), 100000),
            reshuffle_each_iteration=True,
            seed=SEED,
        )

    ds = ds.batch(batch_size, drop_remainder=False)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


# ------------------------------------------------------------
# 8) Base scoring model
# ------------------------------------------------------------
class ItemItemEncoder(tf.keras.Model):
    def __init__(self, num_items: int, embedding_dim: int = 64, l2_reg: float = 1e-6):
        super().__init__()
        self.item_embedding = tf.keras.layers.Embedding(
            input_dim=num_items,
            output_dim=embedding_dim,
            embeddings_initializer="glorot_uniform",
            embeddings_regularizer=tf.keras.regularizers.l2(l2_reg),
            name="item_embedding",
        )
        self.item_bias = tf.keras.layers.Embedding(
            input_dim=num_items,
            output_dim=1,
            embeddings_initializer="zeros",
            name="item_bias",
        )

    def call(self, inputs, training=False):
        src = inputs["src_item"]   # [N]
        dst = inputs["dst_item"]   # [N]

        src_emb = self.item_embedding(src)               # [N, D]
        dst_emb = self.item_embedding(dst)               # [N, D]
        dst_bias = self.item_bias(dst)                   # [N, 1]

        dot = tf.reduce_sum(src_emb * dst_emb, axis=-1, keepdims=True)  # [N, 1]
        logits = dot + dst_bias                                           # [N, 1]
        return logits


# ------------------------------------------------------------
# 9) Training wrapper with custom train_step + test_step
# ------------------------------------------------------------
class BatchNegativeSamplingModel(tf.keras.Model):
    def __init__(
        self,
        scorer: ItemItemEncoder,
        num_items: int,
        neg_probs: np.ndarray,
        negatives_per_positive: int = 3,
        positive_neighbor_sets: Optional[Dict[int, Set[int]]] = None,
        filter_false_negatives: bool = False,
        max_resample_attempts: int = 3,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.scorer = scorer
        self.num_items = int(num_items)
        self.negatives_per_positive = int(negatives_per_positive)
        self.filter_false_negatives = bool(filter_false_negatives)
        self.max_resample_attempts = int(max_resample_attempts)

        neg_probs = np.asarray(neg_probs, dtype=np.float32)
        neg_probs = neg_probs / neg_probs.sum()
        self.neg_probs_np = neg_probs
        self.neg_logits = tf.constant(np.log(neg_probs[None, :] + 1e-12), dtype=tf.float32)

        self.positive_neighbor_sets = positive_neighbor_sets or {}

        # training metrics
        self.train_loss_tracker = tf.keras.metrics.Mean(name="loss")
        self.train_auc_metric = tf.keras.metrics.AUC(name="auc")

        # validation/test metrics
        self.val_loss_tracker = tf.keras.metrics.Mean(name="val_loss")
        self.val_auc_metric = tf.keras.metrics.AUC(name="val_auc")

    @property
    def metrics(self):
        # Keras resets these at the start of each epoch/eval phase
        return [
            self.train_loss_tracker,
            self.train_auc_metric,
            self.val_loss_tracker,
            self.val_auc_metric,
        ]

    def call(self, inputs, training=False):
        return self.scorer(inputs, training=training)

    def _sample_negatives_basic(self, src):
        batch_size_tensor = tf.shape(src)[0]
        neg_dst = tf.random.categorical(
            logits=self.neg_logits,
            num_samples=batch_size_tensor * self.negatives_per_positive,
        )
        neg_dst = tf.reshape(neg_dst, [-1])
        neg_dst = tf.cast(neg_dst, tf.int32)
        neg_src = tf.repeat(src, repeats=self.negatives_per_positive)
        return neg_src, neg_dst

    def _sample_negatives_filtered_numpy(self, src_np: np.ndarray, pos_dst_np: np.ndarray):
        bsz = len(src_np)
        total = bsz * self.negatives_per_positive

        neg_src = np.repeat(src_np, self.negatives_per_positive).astype(np.int32)
        neg_dst = np.random.choice(
            self.num_items,
            size=total,
            replace=True,
            p=self.neg_probs_np,
        ).astype(np.int32)

        repeated_pos_dst = np.repeat(pos_dst_np, self.negatives_per_positive)

        for i in range(total):
            s = int(neg_src[i])
            true_d = int(repeated_pos_dst[i])
            banned = self.positive_neighbor_sets.get(s, set())

            attempts = 0
            while (neg_dst[i] == true_d or neg_dst[i] in banned) and attempts < self.max_resample_attempts:
                neg_dst[i] = np.random.choice(
                    self.num_items,
                    size=1,
                    replace=True,
                    p=self.neg_probs_np,
                )[0]
                attempts += 1

        return neg_src, neg_dst

    def _make_pos_neg_batch(self, src, pos_dst, pos_w):
        if self.filter_false_negatives:
            neg_src, neg_dst = tf.numpy_function(
                func=self._sample_negatives_filtered_numpy,
                inp=[src, pos_dst],
                Tout=[tf.int32, tf.int32],
            )
            neg_src.set_shape([None])
            neg_dst.set_shape([None])
        else:
            neg_src, neg_dst = self._sample_negatives_basic(src)

        all_src = tf.concat([src, neg_src], axis=0)
        all_dst = tf.concat([pos_dst, neg_dst], axis=0)

        labels = tf.concat([
            tf.ones_like(src, dtype=tf.float32),
            tf.zeros_like(neg_src, dtype=tf.float32),
        ], axis=0)
        labels = tf.expand_dims(labels, axis=-1)  # [N, 1]

        sample_weight = tf.concat([
            pos_w,
            tf.ones_like(neg_src, dtype=tf.float32),
        ], axis=0)  # [N]

        return all_src, all_dst, labels, sample_weight

    def _manual_weighted_bce(self, labels, logits, sample_weight):
        per_example_loss = tf.nn.sigmoid_cross_entropy_with_logits(
            labels=labels,
            logits=logits,
        )  # [N, 1]

        per_example_loss = tf.squeeze(per_example_loss, axis=-1)  # [N]
        weighted_loss = per_example_loss * sample_weight          # [N]
        denom = tf.reduce_sum(sample_weight) + 1e-8
        return tf.reduce_sum(weighted_loss) / denom

    def train_step(self, data):
        batch = data
        src = tf.cast(batch["src_item"], tf.int32)
        pos_dst = tf.cast(batch["pos_dst_item"], tf.int32)
        pos_w = tf.cast(batch["pos_weight"], tf.float32)

        all_src, all_dst, labels, sample_weight = self._make_pos_neg_batch(src, pos_dst, pos_w)

        with tf.GradientTape() as tape:
            logits = self.scorer(
                {"src_item": all_src, "dst_item": all_dst},
                training=True,
            )

            loss = self._manual_weighted_bce(
                labels=labels,
                logits=logits,
                sample_weight=sample_weight,
            )

            if self.scorer.losses:
                loss += tf.add_n(self.scorer.losses)

        grads = tape.gradient(loss, self.scorer.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.scorer.trainable_variables))

        probs = tf.sigmoid(logits)

        self.train_loss_tracker.update_state(loss)
        self.train_auc_metric.update_state(labels, probs, sample_weight=sample_weight)

        return {
            "loss": self.train_loss_tracker.result(),
            "auc": self.train_auc_metric.result(),
        }

    def test_step(self, data):
        batch = data
        src = tf.cast(batch["src_item"], tf.int32)
        pos_dst = tf.cast(batch["pos_dst_item"], tf.int32)
        pos_w = tf.cast(batch["pos_weight"], tf.float32)

        all_src, all_dst, labels, sample_weight = self._make_pos_neg_batch(src, pos_dst, pos_w)

        logits = self.scorer(
            {"src_item": all_src, "dst_item": all_dst},
            training=False,
        )

        loss = self._manual_weighted_bce(
            labels=labels,
            logits=logits,
            sample_weight=sample_weight,
        )

        if self.scorer.losses:
            loss += tf.add_n(self.scorer.losses)

        probs = tf.sigmoid(logits)

        self.val_loss_tracker.update_state(loss)
        self.val_auc_metric.update_state(labels, probs, sample_weight=sample_weight)

        return {
            "loss": self.val_loss_tracker.result(),
            "auc": self.val_auc_metric.result(),
        }


# ------------------------------------------------------------
# 10) Train function
# ------------------------------------------------------------
def train_item_item_model(
    edges_df: pd.DataFrame,
    embedding_dim: int = 64,
    batch_size: int = 4096,
    epochs: int = 10,
    negatives_per_positive: int = 3,
    lr: float = 3e-4,              
    weight_decay: float = 1e-4,    
    filter_false_negatives: bool = False,
    max_resample_attempts: int = 3,
    val_frac: float = 0.2,

    negative_sampling_mode: str = "popular",   
    negative_sampling_power: float = 0.75,     
):
    train_edges_df, val_edges_df = train_val_split_edges(
        edges_df=edges_df,
        val_frac=val_frac,
        seed=SEED,
    )

    # Build vocabulary from full graph so train/val share same ids
    item2idx, idx2item = build_item_vocab(edges_df)
    num_items = len(item2idx)

    train_ds = make_positive_tf_dataset(
        edges_df=train_edges_df,
        item2idx=item2idx,
        batch_size=batch_size,
        shuffle=True,
    )

    val_ds = make_positive_tf_dataset(
        edges_df=val_edges_df,
        item2idx=item2idx,
        batch_size=batch_size,
        shuffle=False,
    )

    # Usually build negative distribution from train only
    neg_probs = build_negative_sampling_distribution(
        train_edges_df,
        item2idx,
        mode=negative_sampling_mode,
        power=negative_sampling_power,
    )

    positive_neighbor_sets = None
    if filter_false_negatives:
        positive_neighbor_sets = build_positive_neighbor_sets(train_edges_df, item2idx)

    scorer = ItemItemEncoder(
        num_items=num_items,
        embedding_dim=embedding_dim,
    )

    model = BatchNegativeSamplingModel(
        scorer=scorer,
        num_items=num_items,
        neg_probs=neg_probs,
        negatives_per_positive=negatives_per_positive,
        positive_neighbor_sets=positive_neighbor_sets,
        filter_false_negatives=filter_false_negatives,
        max_resample_attempts=max_resample_attempts,
    )

    lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=lr,
        decay_steps=1000,
        decay_rate=weight_decay,
        staircase=True,
    )

    optimizer = tf.keras.optimizers.Adam(
        learning_rate=lr_schedule,
    )

    model.compile(
        optimizer=optimizer,
        run_eagerly=False,   # set True temporarily if debugging
    )

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        verbose=1,
    )

    return {
        "model": model,
        "scorer": scorer,
        "item2idx": item2idx,
        "idx2item": idx2item,
        "history": history,
        "train_edges_df": train_edges_df,
        "val_edges_df": val_edges_df,
    }

# ------------------------------------------------------------
# 11) Embeddings + retrieval
# ------------------------------------------------------------
def get_item_embeddings(model, normalize=True):
    if hasattr(model, "scorer"):
        emb = model.scorer.item_embedding.get_weights()[0]
    else:
        emb = model.item_embedding.get_weights()[0]

    if normalize:
        emb = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-12)
    return emb


def recommend_for_item(
    anchor_item,
    model,
    item2idx: dict,
    idx2item: dict,
    top_k: int = 20,
    exclude_self: bool = True,
):
    if anchor_item not in item2idx:
        return pd.DataFrame(columns=["product_id", "score"])

    emb = get_item_embeddings(model, normalize=True)
    anchor_idx = item2idx[anchor_item]
    scores = emb @ emb[anchor_idx]

    if exclude_self:
        scores[anchor_idx] = -1e9

    top_k = min(top_k, len(scores))
    top_idx = np.argpartition(-scores, range(top_k))[:top_k]
    top_idx = top_idx[np.argsort(-scores[top_idx])]

    return pd.DataFrame({
        "product_id": [idx2item[i] for i in top_idx],
        "score": scores[top_idx],
    })


def recommend_for_cart(
    cart_items,
    model,
    item2idx: dict,
    idx2item: dict,
    top_k: int = 20,
):
    valid_idxs = [item2idx[x] for x in cart_items if x in item2idx]
    if not valid_idxs:
        return pd.DataFrame(columns=["product_id", "score"])

    emb = get_item_embeddings(model, normalize=True)
    cart_vec = emb[valid_idxs].mean(axis=0)
    cart_vec = cart_vec / (np.linalg.norm(cart_vec) + 1e-12)

    scores = emb @ cart_vec

    for i in valid_idxs:
        scores[i] = -1e9

    top_k = min(top_k, len(scores))
    top_idx = np.argpartition(-scores, range(top_k))[:top_k]
    top_idx = top_idx[np.argsort(-scores[top_idx])]

    return pd.DataFrame({
        "product_id": [idx2item[i] for i in top_idx],
        "score": scores[top_idx],
    })

C:\Users\PC\.conda\envs\recora\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
# import os
# import clickhouse_connect

# CLICKHOUSE_CONFIG = {
# 	"host": os.getenv("CH_HOST", "172.30.36.13"),
# 	"port": int(os.getenv("CH_PORT", "8123")),
# 	"username": os.getenv("CH_USER", "default"),
# 	"password": os.getenv("CH_PASSWORD", "192837465AhurA!@#"),
# 	"database": os.getenv("CH_DATABASE", "KF"),
# 	"secure": os.getenv("CH_SECURE", "false").lower() == "true",
# }

# client = clickhouse_connect.get_client(**CLICKHOUSE_CONFIG)

# activity_type_ids = (1, 29)
# interval = 180
# query_orders = f'''
# SELECT ms.activity_type_id AS activity_type_id,
#         oi.order_id AS order_id, 
#         pp.category_level2_id AS category_id
# FROM KF.fact_order_items oi
#             JOIN KF.plane_products pp
#                 ON oi.shop_product_id = pp.shop_product_id AND
#                 order_date BETWEEN today() - {interval} AND yesterday() AND 
#                 is_gross = 1 AND
#                 product_id != 0 AND
#                 items_voucher_discount = 0
#             JOIN KF.dim_merchant_shops ms
#                 ON ms.merchant_shop_id = pp.merchant_shop_id AND
#                 activity_type_id IN {activity_type_ids} AND
#                 merchant_shop_polygon_id IN (66, 67, 69, 71, 76, 86, 94) AND
#                 category_id NOT IN (
#                 0,
#                 309,
#                 674,
#                 1103,
#                 1416,
#                 1458,
#                 1659,
#                 1676,
#                 1968,
#                 2036,
#                 3063,
#                 3166,
#                 3629,
#                 3911
#                 )
# GROUP BY ms.activity_type_id, oi.order_id, pp.category_level2_id 
# '''

# df_baskets = client.query_df(query_orders)
# client.close()

# # query_categories = f'''
# # SELECT * FROM KF.dim_categories
# # '''

# # df_categories = client.query_df(query_categories)
# # client.close()

In [3]:
# df_baskets.to_csv('./sample_data/baskets.csv', index=False)
# df_categories.to_csv('./sample_data/categories.csv', index=False)

In [42]:
df_baskets = pd.read_csv('./sample_data/baskets.csv')
df_categories = pd.read_csv('./sample_data/categories.csv')

In [43]:
min_support = int(df_baskets['order_id'].nunique() * 0.0025)
min_support

730

In [45]:
# Build graph
edges_df, item_freq = build_copurchase_graph(
    df=df_baskets,
    order_col="order_id",
    item_col="category_id",
    min_item_freq=min_support,
    min_items_per_order=3,
    max_items_per_order=10,
    weight_mode="ppmi",
    topk_neighbors=10,
)


print("\nGraph edges sample:")
edges_df.head(10)


Graph edges sample:


,src_item,dst_item,weight,cosine,lift,shrink,lift_shrink,ppmi,confidence,co-occurance
0,69,81,0.078775,0.159416,1.081961,0.997097,1.078821,0.078775,0.082669,3435
1,69,3070,0.088171,0.241355,1.092175,0.998720,1.090776,0.088171,0.187721,7800
2,69,55,0.099620,0.072406,1.104751,0.985795,1.089058,0.099620,0.016702,694
3,69,58,0.109750,0.087138,1.115999,0.990050,1.104894,0.109750,0.023946,995
4,69,3099,0.116091,0.071463,1.123098,0.985185,1.106459,0.116091,0.016004,665
5,69,70,0.146878,0.244901,1.158213,0.998681,1.156685,0.146878,0.182258,7573
6,69,73,0.148807,0.228014,1.160449,0.998476,1.158680,0.148807,0.157686,6552
7,69,37,0.157386,0.210283,1.170447,0.998193,1.168332,0.157386,0.132969,5525
8,69,68,0.186581,0.231099,1.205122,0.998459,1.203265,0.186581,0.155977,6481
9,69,36,0.341637,0.095712,1.407250,0.989605,1.392621,0.341637,0.022912,952


In [46]:
# edges_df['weight'] = edges_df['lift_shrink']

In [47]:
artifacts = train_item_item_model(
    edges_df=edges_df,
    embedding_dim=128,
    batch_size=16,
    epochs=50,
    negatives_per_positive=1,
    lr=4e-4,
    weight_decay=0.96,
    filter_false_negatives=True,
    max_resample_attempts=5,
    val_frac=0.1,
)

model = artifacts["model"]
item2idx = artifacts["item2idx"]
idx2item = artifacts["idx2item"]

Epoch 1/50
49/49 [==============================] - 3s 24ms/step - loss: 0.7024 - auc: 0.4827 - val_loss: 0.6857 - val_auc: 0.6333
Epoch 2/50
49/49 [==============================] - 1s 13ms/step - loss: 0.6913 - auc: 0.5716 - val_loss: 0.6936 - val_auc: 0.6298
Epoch 3/50
49/49 [==============================] - 1s 14ms/step - loss: 0.6851 - auc: 0.6431 - val_loss: 0.6724 - val_auc: 0.7200
Epoch 4/50
49/49 [==============================] - 1s 14ms/step - loss: 0.6765 - auc: 0.7133 - val_loss: 0.6743 - val_auc: 0.6977
Epoch 5/50
49/49 [==============================] - 1s 13ms/step - loss: 0.6674 - auc: 0.7765 - val_loss: 0.6797 - val_auc: 0.7068
Epoch 6/50
49/49 [==============================] - 1s 13ms/step - loss: 0.6606 - auc: 0.8262 - val_loss: 0.6627 - val_auc: 0.7975
Epoch 7/50
49/49 [==============================] - 1s 14ms/step - loss: 0.6567 - auc: 0.8540 - val_loss: 0.6728 - val_auc: 0.8137
Epoch 8/50
49/49 [==============================] - 1s 13ms/step - loss: 0.6433 - a

In [48]:
# # 0.002 min support

# edges_df, item_freq = build_copurchase_graph(
#     df=df_baskets,
#     order_col="order_id",
#     item_col="category_id",
#     min_item_freq=min_support,
#     min_items_per_order=5,
#     max_items_per_order=10,
#     weight_mode="lift",
#     topk_neighbors=15,
# )


# print("\nGraph edges sample:")
# print(edges_df.head(10))

# artifacts = train_item_item_model(
#     edges_df=edges_df,
#     embedding_dim=64,
#     batch_size=4,
#     epochs=30,
#     negatives_per_positive=5,
#     lr=1e-3,
#     filter_false_negatives=True,
#     max_resample_attempts=5,
#     val_frac=0.2,
# )

# model = artifacts["model"]
# item2idx = artifacts["item2idx"]
# idx2item = artifacts["idx2item"]

In [49]:
df_categories = df_categories[df_categories['type'] == 'leaf']
df_categories['category_id'] = df_categories['category_id'].astype(str)

In [54]:
print("\nRecommendations for item = 'phone'")
recoms = recommend_for_item(
    anchor_item="72",
    model=model,
    item2idx=item2idx,
    idx2item=idx2item,
    top_k=10,
    exclude_self=False
)

recoms.merge(df_categories, left_on='product_id', right_on='category_id', how='left')


Recommendations for item = 'phone'


,product_id,score,category_id,status,type,title
0,72,1.000000,72,active,leaf,کشک
1,3136,0.552195,3136,active,leaf,سبزیجات و صیفی‌‌جات منجمد
2,38,0.416544,38,active,leaf,ادویه و افزودنی
3,3100,0.407265,3100,active,leaf,ماکارونی، رشته و لازانیا
4,3099,0.329046,3099,active,leaf,غلات
5,3133,0.325625,3133,active,leaf,کنسرو سبزیجات و حبوبات
6,3146,0.282033,3146,active,leaf,آبلیمو، سرکه و آبغوره
7,86,0.247197,86,active,leaf,حبوبات و سویا
8,3102,0.237068,3102,active,leaf,آرد و پودر سوخاری
9,3104,0.216285,3104,active,leaf,رب


In [616]:
edges_df[(edges_df['src_item'] == "73")].merge(df_categories, left_on='dst_item', right_on='category_id', how='left').sort_values(by='weight', ascending=False).head(50)

,src_item,dst_item,weight,cosine,lift,shrink,lift_shrink,ppmi,confidence,co-occurance,category_id,status,type,title
9,73,76,1.545099,0.037767,1.545099,0.931034,1.438540,0.435088,0.006793,135,76,active,leaf,مرغ
8,73,3143,1.440329,0.056923,1.440329,0.970501,1.397841,0.364872,0.016556,329,3143,active,leaf,خرما
7,73,82,1.357468,0.061916,1.357468,0.976359,1.325376,0.305621,0.020783,413,82,active,leaf,میوه
6,73,3070,1.269482,0.194008,1.269482,0.997699,1.266561,0.238609,0.218196,4336,3070,active,leaf,پنیر صبحانه
5,73,70,1.250685,0.182886,1.250685,0.997450,1.247496,0.223692,0.196810,3911,70,active,leaf,ماست
4,73,74,1.225470,0.069233,1.225470,0.982818,1.204414,0.203325,0.028784,572,74,active,leaf,سوسیس و کالباس
3,73,89,1.200213,0.143010,1.200213,0.996003,1.195416,0.182499,0.125403,2492,89,active,leaf,نان
2,73,86,1.198962,0.058189,1.198962,0.976359,1.170618,0.181457,0.020783,413,86,active,leaf,حبوبات و سویا
1,73,3103,1.187276,0.055762,1.187276,0.974555,1.157065,0.171661,0.019273,383,3103,active,leaf,روغن سرخ کردنی
0,73,3101,1.177713,0.069916,1.177713,0.983793,1.158625,0.163574,0.030545,607,3101,active,leaf,روغن آشپزی و سالاد


In [114]:
print("\nRecommendations for cart = ['phone', 'case']")
recoms = recommend_for_cart(
    cart_items=["70", "3085"],
    model=model,
    item2idx=item2idx,
    idx2item=idx2item,
    top_k=10,
)

recoms.merge(df_categories, left_on='product_id', right_on='category_id', how='left')


Recommendations for cart = ['phone', 'case']


,product_id,score,category_id,status,type,title
0,137,0.623532,137,active,leaf,پفک و اسنک
1,3121,0.560702,3121,active,leaf,تخمه و مغزهای طعم‌دار
2,3091,0.545295,3091,active,leaf,آبمیوه
3,3083,0.495092,3083,active,leaf,پاستیل و مارشمالو
4,32,0.489123,32,active,leaf,لواشک و آلوچه
5,3088,0.487205,3088,active,leaf,پاپ‌کورن
6,3097,0.391005,3097,active,leaf,نوشابه
7,37,0.389918,37,active,leaf,کیک و کلوچه
8,33,0.365453,33,active,leaf,ویفر و بیسکویت
9,101,0.348310,101,active,leaf,نوشیدنی انرژی‌زا
